# VocalCoach — Final Training Plan (P0–P4)

Implements the runs identified in `TRAINING_PLAN.md`. All runs use the **TCN-256-attn2** architecture.

**Stage-1 backbone** (pre-trained, frozen in probe runs): `stage1_tcn_256_conformerattn2_lowlr`  
→ offRPA 99.2%, VDR 78.8%, Med¢ 2.1, best OOD VDR (Vocadito 39.5%)  
→ Was trained on `data/merged_pitchvad` (GTSinger + VocalSet, 51,627 clips)

**Run order:**
| Priority | Run | Type | Depends on |
|---|---|---|---|
| P0 | `stage2_tcn256attn2_deepprobe` | Probe + deep head | backbone ckpt |
| P1 | `stage2_tcn256attn2_curriculum` | Joint + curriculum | backbone ckpt |
| P3 | `stage1_tcn256attn2_notehead` | Stage-1 + note head | backbone ckpt |
| P2-V3 | `stage2_quality_v3_distill` | Quality distill | P0 or P1 winner |
| P2-V2 | `stage2_quality_v2_ccmusic` | Quality 9-dim | V3 ckpt |
| P2-V1 | `stage2_quality_v1_contrastive` | Quality contrastive | quality_pairs_50k |
| P4 | OOD eval on P0/P1 winners | Eval only | P0, P1 ckpts |

P0, P1, P3 can all run **in parallel** (independent checkpoints). P2 variants run sequentially after.

---

### Data file guide

| File | Contents | Used by |
|---|---|---|
| `data/clean.npz` | GTSinger only — 40,940 clips | P0 eval (probe mode), P2 quality |
| `data/merged_pitchvad/clean.npz` | GTSinger + VocalSet — 51,627 clips | P1, P3 (joint training, matches backbone training data) |
| `data/noise.npz` | MUSAN noise pool | all runs via `--noise-dir data` |
| `data/test.npz` | GTSinger held-out eval set | all runs via `--eval-dir data` |

P0 (probe): `--data-dir /content/data` — `clean.npz` used only for eval, `noise.npz`+`test.npz` auto-found.  
P1, P3 (joint): `--data-dir /content/data/merged_pitchvad --noise-dir /content/data --eval-dir /content/data` — merged training data, eval+noise from parent.

---

### Drive layout expected
```
My Drive/musicalAI/vocalCoach/
  NanoPitch_data_plan.zip          ← new zip (see upload instructions below)
  NanoPitch-runs/
    stage1_tcn_256_conformerattn2_lowlr/   ← Stage-1 backbone (upload best_metric.pth via Cell 1b)
    stage2_tcn256attn2_deepprobe/
    stage2_tcn256attn2_curriculum/
    stage1_tcn256attn2_notehead/
    stage2_quality_v3_distill/
    stage2_quality_v2_ccmusic/
    stage2_quality_v1_contrastive/
```

### New files to upload to Drive (one-time from local)
```bash
cd ~/NanoPitch-MusicalAI
zip -1 NanoPitch_data_plan.zip \
    data/clean.npz \
    data/noise.npz \
    data/test.npz \
    data/merged_pitchvad/clean.npz \
    data/vocalset/technique_train.npz \
    data/vocalset/technique_test.npz \
    data/annotated_vocalset/note_train.npz \
    data/annotated_vocalset/note_test.npz \
    data/gtsinger_technique/technique_train.npz \
    data/gtsinger_technique/technique_gtsinger_train.npz \
    data/gtsinger_technique/technique_gtsinger_test.npz \
    data/quality/quality_mse.npz \
    data/quality/quality_ccmusic.npz \
    data/quality_50k/quality_pairs.npz
```
Upload `NanoPitch_data_plan.zip` to `My Drive/musicalAI/vocalCoach/`.

### Checkpoint upload (for --resume)
The backbone checkpoint is uploaded separately via Cell 1b (not in the zip).  
File: `vocalcoach/runs/stage1_tcn_256_conformerattn2_lowlr/checkpoints/best_metric.pth`

## Cell 1 — GPU check + batch size

In [ ]:
!nvidia-smi
import torch
print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU:            {torch.cuda.get_device_name(0)}')
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM:           {vram_gb:.1f} GB')

BATCH_SIZE   = 128 if vram_gb > 30 else 64 if vram_gb > 15 else 32
NUM_WORKERS  = 8
print(f'\nUsing batch_size={BATCH_SIZE}, num_workers={NUM_WORKERS}')

## Cell 1b — Upload backbone checkpoint (if not already on Drive)

Run this cell **once** to upload `best_metric.pth` for `stage1_tcn_256_conformerattn2_lowlr`  
from your local machine. Skip if the file is already in Drive.

In [ ]:
import os
from google.colab import files

CKPT_DEST = '/content/drive/MyDrive/musicalAI/vocalCoach/NanoPitch-runs/stage1_tcn_256_conformerattn2_lowlr/checkpoints'
ckpt_path = f'{CKPT_DEST}/best_metric.pth'

if os.path.exists(ckpt_path):
    print(f'Checkpoint already on Drive: {ckpt_path}')
    print('Skip this cell.')
else:
    print('Upload best_metric.pth from:')
    print('  vocalcoach/runs/stage1_tcn_256_conformerattn2_lowlr/checkpoints/best_metric.pth')
    uploaded = files.upload()
    for fname, data in uploaded.items():
        os.makedirs(CKPT_DEST, exist_ok=True)
        dest = os.path.join(CKPT_DEST, 'best_metric.pth')
        with open(dest, 'wb') as f:
            f.write(data)
        size_mb = len(data) / 1e6
        print(f'Saved {fname} ({size_mb:.1f} MB) → {dest}')

## Cell 2 — Mount Drive and extract data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os, numpy as np

DRIVE_ROOT = '/content/drive/MyDrive/musicalAI/vocalCoach'
RUNS_DIR   = f'{DRIVE_ROOT}/NanoPitch-runs'
os.makedirs(RUNS_DIR, exist_ok=True)

# All files extracted from NanoPitch_data_plan.zip → /content/
# Two clean.npz files serve different purposes:
#   /content/data/clean.npz              = GTSinger only (40,940 clips) — eval for probe runs
#   /content/data/merged_pitchvad/clean.npz = GTSinger+VocalSet (51,627 clips) — joint training (P1, P3)
files_needed = {
    'data/clean.npz':                                          '/content/data/clean.npz',
    'data/noise.npz':                                          '/content/data/noise.npz',
    'data/test.npz':                                           '/content/data/test.npz',
    'data/merged_pitchvad/clean.npz':                          '/content/data/merged_pitchvad/clean.npz',
    'data/vocalset/technique_train.npz':                       '/content/data/vocalset/technique_train.npz',
    'data/vocalset/technique_test.npz':                        '/content/data/vocalset/technique_test.npz',
    'data/annotated_vocalset/note_train.npz':                  '/content/data/annotated_vocalset/note_train.npz',
    'data/annotated_vocalset/note_test.npz':                   '/content/data/annotated_vocalset/note_test.npz',
    'data/gtsinger_technique/technique_train.npz':             '/content/data/gtsinger_technique/technique_train.npz',
    'data/gtsinger_technique/technique_gtsinger_train.npz':    '/content/data/gtsinger_technique/technique_gtsinger_train.npz',
    'data/gtsinger_technique/technique_gtsinger_test.npz':     '/content/data/gtsinger_technique/technique_gtsinger_test.npz',
    'data/quality/quality_mse.npz':                            '/content/data/quality/quality_mse.npz',
    'data/quality/quality_ccmusic.npz':                        '/content/data/quality/quality_ccmusic.npz',
    'data/quality_50k/quality_pairs.npz':                      '/content/data/quality_50k/quality_pairs.npz',
}

def _npz_valid(path):
    if not os.path.exists(path): return False
    try:
        with zipfile.ZipFile(path, 'r'): return True
    except zipfile.BadZipFile:
        return False

missing = []
for name, dest in files_needed.items():
    if not _npz_valid(dest):
        if os.path.exists(dest):
            print(f'  CORRUPTED (re-extracting): {dest}')
            os.remove(dest)
        missing.append(name)

if missing:
    zip_path = f'{DRIVE_ROOT}/NanoPitch_data_plan.zip'
    print(f'Extracting {len(missing)} file(s) from NanoPitch_data_plan.zip ...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        for name in missing:
            dest = files_needed[name]
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            z.extract(name, '/content/')
            if _npz_valid(dest):
                print(f'  OK  {name}')
            else:
                raise RuntimeError(f'Extraction failed: {dest}')
else:
    print('All data files present.')

!ls -lh /content/data/
!ls -lh /content/data/merged_pitchvad/
!ls -lh /content/data/quality/ 2>/dev/null
!ls -lh /content/data/quality_50k/ 2>/dev/null

## Cell 3 — Clone repo and install dependencies

In [ ]:
import os

REPO_DIR = '/content/NanoPitch-MusicalAI'
REPO_URL = 'https://github.com/rajat17-personal/NanoPitch-MusicalAI'
BRANCH   = 'feat/finalProject'

if os.path.isdir(f'{REPO_DIR}/.git'):
    print('Repo present — pulling latest...')
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull --ff-only origin {BRANCH}
else:
    print('Cloning...')
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -r requirements.txt --quiet
print('Setup complete.')

## Cell 4 — Verify data and restore backbone checkpoint

In [ ]:
import numpy as np, os

def check(label, path, key='lengths'):
    try:
        d = np.load(path)
        n = d[key].shape[0]
        print(f'  OK  {label:<55} {n:>6} clips')
    except Exception as e:
        print(f'  MISSING  {label:<52} {e}')

print('=== Pitch/VAD data ===')
check('clean.npz',                                '/content/data/clean.npz')
check('noise.npz',                                '/content/data/noise.npz')
check('test.npz',                                 '/content/data/test.npz', key='clips')

print('\n=== Technique data ===')
check('vocalset/technique_train.npz',             '/content/data/vocalset/technique_train.npz')
check('vocalset/technique_test.npz',              '/content/data/vocalset/technique_test.npz')
check('gtsinger_technique/technique_train.npz',   '/content/data/gtsinger_technique/technique_train.npz')
check('gtsinger_technique/gtsinger_train.npz',    '/content/data/gtsinger_technique/technique_gtsinger_train.npz')

print('\n=== Note head data ===')
check('annotated_vocalset/note_train.npz',        '/content/data/annotated_vocalset/note_train.npz')
check('annotated_vocalset/note_test.npz',         '/content/data/annotated_vocalset/note_test.npz')

print('\n=== Quality head data ===')
check('quality/quality_mse.npz',                  '/content/data/quality/quality_mse.npz', key='lengths')
check('quality/quality_ccmusic.npz',              '/content/data/quality/quality_ccmusic.npz', key='lengths')
# quality_pairs uses lengths_pro not lengths
try:
    d = np.load('/content/data/quality_50k/quality_pairs.npz')
    n = d['lengths_pro'].shape[0]
    print(f'  OK  quality_50k/quality_pairs.npz{"":>22} {n:>6} pairs')
except Exception as e:
    print(f'  MISSING  quality_50k/quality_pairs.npz  {e}')

print('\n=== Backbone checkpoint ===')
BACKBONE_CKPT = f'{RUNS_DIR}/stage1_tcn_256_conformerattn2_lowlr/checkpoints/best_metric.pth'
if os.path.exists(BACKBONE_CKPT):
    sz = os.path.getsize(BACKBONE_CKPT)/1e6
    print(f'  OK  {BACKBONE_CKPT} ({sz:.0f} MB)')
else:
    print(f'  MISSING  {BACKBONE_CKPT}')
    print('  → Run Cell 1b to upload it.')

## Cell 5 — Drive helpers (save / restore)

In [ ]:
import shutil, os

DRIVE_ROOT = '/content/drive/MyDrive/musicalAI/vocalCoach'
RUNS_DIR   = f'{DRIVE_ROOT}/NanoPitch-runs'

# Backbone checkpoint restored to /content/runs/ for fast local I/O during training
BACKBONE_RUN  = 'stage1_tcn_256_conformerattn2_lowlr'
BACKBONE_CKPT = f'{RUNS_DIR}/{BACKBONE_RUN}/checkpoints/best_metric.pth'

def restore_backbone():
    dst = f'/content/runs/{BACKBONE_RUN}/checkpoints/best_metric.pth'
    if os.path.exists(dst):
        print(f'Backbone already at {dst}')
        return dst
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(BACKBONE_CKPT, dst)
    print(f'Restored backbone → {dst}')
    return dst

def restore_from_drive(run_name, ckpt_name='best_metric.pth'):
    src = f'{RUNS_DIR}/{run_name}/checkpoints/{ckpt_name}'
    dst_dir = f'/content/runs/{run_name}/checkpoints'
    os.makedirs(dst_dir, exist_ok=True)
    shutil.copy2(src, f'{dst_dir}/{ckpt_name}')
    print(f'Restored {run_name}/{ckpt_name} ← Drive')
    return f'{dst_dir}/{ckpt_name}'

def save_to_drive(run_name):
    src = f'/content/runs/{run_name}'
    dst = f'{RUNS_DIR}/{run_name}'
    os.makedirs(RUNS_DIR, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'Saved {run_name} → Drive')

# Restore backbone immediately so all training cells can reference it
backbone_ckpt = restore_backbone()
print(f'\nBackbone ready: {backbone_ckpt}')

---
## P0 — Deep-Head Technique Probe

**What's new vs the flat-Linear probe already running:**  
`--deep-technique-head` replaces `Linear(256→5)` with `Linear(256→128)→GELU→Dropout→Linear(128→5)`.  
This is the single most-cited fix for the probe-mode mF1 gap in the proposal (Open Q #2).  
Every prior probe run used a flat Linear — this has never been tried.

**Expected:** mF1 +10–15 points vs flat probe. VDR should be unchanged (backbone frozen).

**Note:** `--data-dir /content/data` points to the root with `clean.npz` + `noise.npz` + `test.npz`.  
In probe mode the pitch/VAD heads are frozen so `clean.npz` is used only for eval, not training.

In [ ]:
backbone_ckpt = f'/content/runs/{BACKBONE_RUN}/checkpoints/best_metric.pth'

!python vocalcoach/train.py \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset /content/data/annotated_vocalset \
    --arch tcn --hidden 256 --n-blocks 8 --n-attn-layers 2 --n-heads 4 \
    --seq-len 600 --epochs 100 --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} \
    --probe-mode --deep-technique-head \
    --w-vad 0.05 --w-pitch 2 --w-technique 1 \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --patience 30 \
    --resume {backbone_ckpt} \
    --output-dir /content/runs/stage2_tcn256attn2_deepprobe

In [ ]:
save_to_drive('stage2_tcn256attn2_deepprobe')

---
## P1 — Curriculum Joint Run

**What's new:** `--curriculum` has **never been run** (0 of 46 runs). It zeros the technique loss  
for epochs 1–30, then linearly ramps it to `--w-technique 1` over 10 epochs. Pitch/VAD converge  
first before technique gradients compete — directly addresses the VDR collapse mechanism.

**Success criterion:** VDR > 75% AND mF1 > 0.60.  
If it beats Run 25 (`stage2_w1_joint_difflr`: mF1=0.769, VDR=79.0%) → new best joint run.

Can run **in parallel** with P0 (independent output dir, same backbone).

In [ ]:
backbone_ckpt = f'/content/runs/{BACKBONE_RUN}/checkpoints/best_metric.pth'

# --data-dir points to merged_pitchvad (GTSinger+VocalSet) to match what the backbone was
# trained on. noise.npz and test.npz live in the parent data/ dir, so we pass them explicitly.
!python vocalcoach/train.py \
    --data-dir /content/data/merged_pitchvad \
    --noise-dir /content/data \
    --eval-dir /content/data \
    --technique-dirs /content/data/vocalset /content/data/annotated_vocalset \
    --arch tcn --hidden 256 --n-blocks 8 --n-attn-layers 2 --n-heads 4 \
    --seq-len 600 --epochs 120 --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} \
    --curriculum --curriculum-warmup 30 --curriculum-ramp 10 \
    --w-vad 0.05 --w-pitch 2 --w-technique 1 \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --patience 30 \
    --resume {backbone_ckpt} \
    --output-dir /content/runs/stage2_tcn256attn2_curriculum

In [ ]:
save_to_drive('stage2_tcn256attn2_curriculum')

---
## P3 — Note Segmentation Head (Variant 4)

Adds `head_note_onset` + `head_note_offset` — two binary frame-level classifiers.  
Trains jointly with pitch/VAD (backbone is **not** frozen — note onset correlates with F0 transients).  

**Data:** `data/annotated_vocalset/note_train.npz` — 52 MB, 824 clips, 18,412 annotated notes.  

**Unlocks coaching metrics** without post-hoc DSP: note-level pitch accuracy, intonation drift  
(cents per note), attack speed (frames to target ±50¢), pitch stability within note.

Can run **in parallel** with P0 and P1 — independent of technique training.

In [ ]:
import numpy as np
note_path = '/content/data/annotated_vocalset/note_train.npz'
d = np.load(note_path)
print(f'note_train.npz keys: {list(d.keys())}')
print(f'clips:        {d["lengths"].shape[0]}')
print(f'total frames: {d["mel"].shape[0]:,}')

# Two supported layouts — NoteDataset handles both automatically (no crash, no zeros)
if 'onset' in d and 'offset' in d:
    n_on = int(d['onset'].sum())
    print(f'Layout: DENSE  — binary onset/offset arrays present ✓  ({n_on:,} onset frames)')
elif 'note_onsets' in d and 'note_offsets' in d:
    n = len(d['note_onsets'])
    print(f'Layout: SPARSE — note_onsets/note_offsets present ✓  ({n:,} notes → expanded to dense at load time)')
else:
    print('Layout: UNKNOWN — neither dense nor sparse keys found ✗')
    print('  Expected: onset+offset (dense) OR note_onsets+note_offsets (sparse)')

In [ ]:
backbone_ckpt = f'/content/runs/{BACKBONE_RUN}/checkpoints/best_metric.pth'

# Same data-dir logic as P1: merged_pitchvad for training, noise+eval from parent data/
!python vocalcoach/train.py \
    --data-dir /content/data/merged_pitchvad \
    --noise-dir /content/data \
    --eval-dir /content/data \
    --technique-dirs /content/data/vocalset /content/data/annotated_vocalset \
    --note-head --note-dirs /content/data/annotated_vocalset --w-note 0.5 \
    --arch tcn --hidden 256 --n-blocks 8 --n-attn-layers 2 --n-heads 4 \
    --seq-len 600 --epochs 120 --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} \
    --lr 0.001 \
    --w-vad 0.05 --w-pitch 2 \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --patience 30 \
    --resume {backbone_ckpt} \
    --output-dir /content/runs/stage1_tcn256attn2_notehead

In [ ]:
save_to_drive('stage1_tcn256attn2_notehead')

---
## P4 — OOD Eval (gate before deploying Stage-2)

Run after P0 and P1 complete. No Stage-2 technique checkpoint has ever been OOD-evaluated.  
Acceptance: VDR and RPA must not drop >5pp relative to in-dist scores.  
Best in-dist reference: Run 45 VDR=78.8%, RPA=99.2%.

In [ ]:
import os

# Restore whichever Stage-2 checkpoints you want to gate
for run in ['stage2_tcn256attn2_deepprobe', 'stage2_tcn256attn2_curriculum']:
    ckpt = f'/content/runs/{run}/checkpoints/best_metric.pth'
    if not os.path.exists(ckpt):
        restore_from_drive(run)

    print(f'\n=== OOD eval: {run} ===')
    !python scripts/evalOOD.py \
        --checkpoint /content/runs/{run}/checkpoints/best_metric.pth \
        --dataset vocadito \
        --data-dir /content/data/vocadito \
        --log results/ood_log.json \
        --regression-threshold 0.05

---
## P2 — Quality Scoring Head

Run **after** P0/P1 complete and OOD gate passes. The quality head uses `--probe-mode`:  
backbone + pitch/VAD/technique heads are ALL FROZEN — only `head_quality` trains.

**Order:** V3 (distill) → V2 (9-dim ccmusic) → V1 (contrastive)

Set `P2_BASE_RUN` to whichever P0/P1 winner has best VDR+mF1 compound.

In [ ]:
# Update this after P0 and P1 results are known
P2_BASE_RUN = 'stage2_tcn256attn2_deepprobe'   # replace with actual winner

p2_ckpt = f'/content/runs/{P2_BASE_RUN}/checkpoints/best_metric.pth'
if not os.path.exists(p2_ckpt):
    restore_from_drive(P2_BASE_RUN)
print(f'P2 base: {p2_ckpt}')

### P2-V3 — SingMOS-Pro distillation (train first: smallest, no contrastive dependency)

Stage 1 (epochs 1–30): MSE on SingMOS-Pro AudioScore pseudo-labels → calibrates scale.  
Stage 2 (epochs 31+): ranking loss on PopBuTFy pairs → anchors amateur/pro separation.

In [ ]:
!python vocalcoach/train.py \
    --arch tcn --hidden 256 --n-blocks 8 --n-attn-layers 2 --n-heads 4 \
    --seq-len 600 --epochs 60 --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} \
    --probe-mode \
    --quality-variant 3 \
    --quality-mse-npz /content/data/quality/quality_mse.npz \
    --quality-pairs-npz /content/data/quality_50k/quality_pairs.npz \
    --quality-epochs-mse 30 \
    --w-quality-mse 1.0 --w-ranking 1.0 --ranking-margin 0.5 \
    --patience 20 \
    --resume {p2_ckpt} \
    --output-dir /content/runs/stage2_quality_v3_distill

In [ ]:
save_to_drive('stage2_quality_v3_distill')

### P2-V2 — 9-dim ccmusic expert labels (resume from V3)

Adds ccmusic 9-dim expert supervision on top of the V3-initialised head.  
9 output dims: Pitch / Rhythm / Timbre / Breath / Vibrato / Dynamic / Pronunciation / Vocal Range / Overall.  
Maps directly to the radar chart in the coaching demo.

In [ ]:
v3_ckpt = '/content/runs/stage2_quality_v3_distill/checkpoints/best_metric.pth'
if not os.path.exists(v3_ckpt):
    restore_from_drive('stage2_quality_v3_distill')

!python vocalcoach/train.py \
    --arch tcn --hidden 256 --n-blocks 8 --n-attn-layers 2 --n-heads 4 \
    --seq-len 600 --epochs 60 --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} \
    --probe-mode \
    --quality-variant 2 \
    --quality-mse-npz /content/data/quality/quality_mse.npz \
    --quality-ccmusic-npz /content/data/quality/quality_ccmusic.npz \
    --quality-pairs-npz /content/data/quality_50k/quality_pairs.npz \
    --quality-epochs-mse 30 \
    --w-quality-mse 1.0 --w-ranking 1.0 --ranking-margin 0.5 \
    --patience 20 \
    --resume {v3_ckpt} \
    --output-dir /content/runs/stage2_quality_v2_ccmusic

In [ ]:
save_to_drive('stage2_quality_v2_ccmusic')

### P2-V1 — Contrastive-only ranking (PopBuTFy pairs)

Scalar head trained purely on PopBuTFy pro/amateur ranking pairs.  
No MSE pre-training — simpler but needs the contrastive signal to be strong enough alone.  
Good ablation baseline vs V3 (distill + contrastive).

In [ ]:
!python vocalcoach/train.py \
    --arch tcn --hidden 256 --n-blocks 8 --n-attn-layers 2 --n-heads 4 \
    --seq-len 600 --epochs 60 --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} \
    --probe-mode \
    --quality-variant 1 \
    --quality-pairs-npz /content/data/quality_50k/quality_pairs.npz \
    --w-ranking 1.0 --ranking-margin 0.5 \
    --patience 20 \
    --resume {p2_ckpt} \
    --output-dir /content/runs/stage2_quality_v1_contrastive

In [ ]:
save_to_drive('stage2_quality_v1_contrastive')

---
## Download all new checkpoints for local eval

Zips only `best_metric.pth` from each completed run and downloads to your machine.  
Extract with `unzip plan_checkpoints.zip` from `~/NanoPitch-MusicalAI/`.

In [ ]:
import os, zipfile
from google.colab import files

# Update this list after runs complete
RUN_NAMES = [
    'stage2_tcn256attn2_deepprobe',
    'stage2_tcn256attn2_curriculum',
    'stage1_tcn256attn2_notehead',
    'stage2_quality_v3_distill',
    'stage2_quality_v2_ccmusic',
    'stage2_quality_v1_contrastive',
]

zip_path = '/content/plan_checkpoints.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for run in RUN_NAMES:
        for ckpt in ['best_metric.pth', 'best_loss.pth']:
            src = f'/content/runs/{run}/checkpoints/{ckpt}'
            if not os.path.exists(src):
                print(f'  SKIP (not found): {run}/{ckpt}')
                continue
            arcname = f'vocalcoach/runs/{run}/checkpoints/{ckpt}'
            zf.write(src, arcname=arcname)
            print(f'  + {arcname}  ({os.path.getsize(src)/1e6:.1f} MB)')

total_mb = os.path.getsize(zip_path)/1e6
print(f'\nZip: {zip_path}  ({total_mb:.1f} MB)')
files.download(zip_path)